In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:

# !pip install networkx
# !pip install node2vec
# !pip install "numpy<2.0" "scipy<1.13" "gensim==4.3.3" node2vec --quiet

# STEP 1: L0 - Graph-based Candidate Generation

In [3]:
import networkx as nx

# Generate graph similar to social network
G = nx.barabasi_albert_graph(5000, 5)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 5000
Edges: 24975


In [4]:
user = 5

print("Neighbors:", list(G.neighbors(user))[:10])

Neighbors: [0, 6, 8, 13, 26, 28, 36, 71, 83, 100]


In [5]:
def graph_candidates(G, user):
    cands = set()

    # Direct neighbors
    neighbors = set(G.neighbors(user))

    for nbr in neighbors:
        for fof in G.neighbors(nbr):
            # Exclude:
            # 1. self
            # 2. already connected users
            if fof != user and fof not in neighbors:
                cands.add(fof)

    return list(cands)

In [6]:
candidates = graph_candidates(G, user)

print("Total candidates:", len(candidates))
print("Sample:", candidates[:10])

Total candidates: 1359
Sample: [1, 2, 3, 4, 7, 9, 10, 11, 12, 14]


# STEP 1: L0 - Graph-based Candidate Generation

In [7]:
from node2vec import Node2Vec

# Node2Vec expects graph with string nodes OR handles int fine (we'll convert later)

node2vec = Node2Vec(
    G,
    dimensions=64,
    walk_length=10,
    num_walks=50,
    workers=2
)

model_emb = node2vec.fit(window=5, min_count=1, batch_words=4)

Computing transition probabilities:   0%|          | 0/5000 [00:00<?, ?it/s]

Generating walks (CPU: 2): 100%|██████████| 25/25 [00:07<00:00,  3.24it/s]


# STEP 3: Heuristic Candidates + Combine ALL (FINAL L0)

In [8]:
import random

def heuristic_candidates(G, user, k=100):
    all_nodes = list(G.nodes())
    
    # remove self + neighbors
    neighbors = set(G.neighbors(user))
    excluded = neighbors.union({user})
    
    pool = list(set(all_nodes) - excluded)
    
    return random.sample(pool, min(k, len(pool)))

In [9]:
heur_cands = heuristic_candidates(G, user)

print("Heuristic candidates:", len(heur_cands))
print("Sample:", heur_cands[:10])

Heuristic candidates: 100
Sample: [258, 3810, 1185, 4578, 4012, 3484, 2283, 3521, 4029, 311]


In [12]:
def embedding_candidates(user, topn=100):
    try:
        similar = model_emb.wv.most_similar(str(user), topn=topn)
        return [int(node) for node, _ in similar]
    except KeyError:
        return []

In [13]:
def generate_candidates(G, user):
    c1 = graph_candidates(G, user)
    c2 = embedding_candidates(user)
    c3 = heuristic_candidates(G, user)

    combined = list(set(c1 + c2 + c3))
    
    return combined

In [14]:
all_cands = generate_candidates(G, user)

print("Total L0 candidates:", len(all_cands))
print("Sample:", all_cands[:10])

Total L0 candidates: 1492
Sample: [1, 2, 3, 4, 7, 9, 10, 11, 12, 14]


# STEP 4: L1 Ranker (Feature Engineering + Training Data)

In [15]:
def features_L1(G, u, v):
    u_neighbors = set(G.neighbors(u))
    v_neighbors = set(G.neighbors(v))
    
    # Mutual friends
    common = len(u_neighbors & v_neighbors)
    
    # Degree
    deg_u = len(u_neighbors)
    deg_v = len(v_neighbors)
    
    # Jaccard
    union = len(u_neighbors | v_neighbors)
    jaccard = common / union if union != 0 else 0
    
    return [common, deg_u, deg_v, jaccard]

In [16]:
import random

def create_l1_dataset(G, num_samples=10000):
    X = []
    y = []
    
    nodes = list(G.nodes())
    
    # Positive samples (existing edges)
    edges = list(G.edges())
    pos_samples = random.sample(edges, min(len(edges), num_samples // 2))
    
    for u, v in pos_samples:
        X.append(features_L1(G, u, v))
        y.append(1)
    
    # Negative samples (non-edges)
    neg_samples = 0
    while neg_samples < num_samples // 2:
        u = random.choice(nodes)
        v = random.choice(nodes)
        
        if u != v and not G.has_edge(u, v):
            X.append(features_L1(G, u, v))
            y.append(0)
            neg_samples += 1
    
    return X, y

In [17]:
X_train, y_train = create_l1_dataset(G)

print("Samples:", len(X_train))
print("Feature example:", X_train[0])
print("Label example:", y_train[0])

Samples: 10000
Feature example: [0, 18, 5, 0.0]
Label example: 1


In [18]:
print("Positive:", sum(y_train))
print("Negative:", len(y_train) - sum(y_train))

Positive: 5000
Negative: 5000


# STEP 5: Train L1 Model + Rank Candidates

In [19]:
from xgboost import XGBClassifier

l1_model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss'
)

l1_model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:28:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [20]:
train_preds = l1_model.predict(X_train)

from sklearn.metrics import accuracy_score

print("Train Accuracy:", accuracy_score(y_train, train_preds))

Train Accuracy: 0.7869


In [21]:
def l1_rank(G, model, user, candidates, top_k=100):
    scored = []
    
    for c in candidates:
        f = features_L1(G, user, c)
        score = model.predict_proba([f])[0][1]
        scored.append((c, score))
    
    # sort by score descending
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored[:top_k]

In [22]:
l1_results = l1_rank(G, l1_model, user, all_cands)

print("Top L1 results:", len(l1_results))
print("Sample:", l1_results[:5])

Top L1 results: 100
Sample: [(20, 0.9872649), (28, 0.9872649), (53, 0.9872649), (24, 0.98689485), (1, 0.9861731)]


In [23]:
# scores should be sorted
print(all(l1_results[i][1] >= l1_results[i+1][1] for i in range(len(l1_results)-1)))

True


# STEP 6: L2 Ranker (Neural Network + Embeddings)

In [24]:
import numpy as np

def features_L2(u, v):
    emb_u = model_emb.wv[str(u)]
    emb_v = model_emb.wv[str(v)]
    
    return np.concatenate([emb_u, emb_v])  # 64 + 64 = 128

In [25]:
f = features_L2(user, l1_results[0][0])
print("Feature length:", len(f))  # should be 128

Feature length: 128


In [26]:
def create_l2_dataset(G, num_samples=10000):
    X = []
    y = []
    
    nodes = list(G.nodes())
    edges = list(G.edges())
    
    # Positive samples
    pos_samples = random.sample(edges, min(len(edges), num_samples // 2))
    
    for u, v in pos_samples:
        X.append(features_L2(u, v))
        y.append(1)
    
    # Negative samples
    neg_samples = 0
    while neg_samples < num_samples // 2:
        u = random.choice(nodes)
        v = random.choice(nodes)
        
        if u != v and not G.has_edge(u, v):
            X.append(features_L2(u, v))
            y.append(0)
            neg_samples += 1
    
    return np.array(X), np.array(y)

In [27]:
X_l2, y_l2 = create_l2_dataset(G)

print("L2 shape:", X_l2.shape)

L2 shape: (10000, 128)


In [28]:
import torch
import torch.nn as nn

class RankNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.fc(x)

l2_model = RankNet()

In [29]:
X_tensor = torch.tensor(X_l2).float()
y_tensor = torch.tensor(y_l2).float().view(-1, 1)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(l2_model.parameters(), lr=0.001)

In [30]:
epochs = 5

for epoch in range(epochs):
    optimizer.zero_grad()
    
    outputs = l2_model(X_tensor)
    loss = criterion(outputs, y_tensor)
    
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6935
Epoch 2, Loss: 0.6921
Epoch 3, Loss: 0.6908
Epoch 4, Loss: 0.6895
Epoch 5, Loss: 0.6882


In [31]:
def l2_rank(model, user, l1_results, top_k=20):
    scored = []
    
    for c, _ in l1_results:
        f = torch.tensor(features_L2(user, c)).float()
        score = model(f).item()
        scored.append((c, score))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    
    return scored[:top_k]

In [32]:
l2_results = l2_rank(l2_model, user, l1_results)

print("Top L2 results:", len(l2_results))
print("Sample:", l2_results[:5])

Top L2 results: 20
Sample: [(28, 0.5719471573829651), (20, 0.5696403980255127), (111, 0.5648233890533447), (35, 0.5603176355361938), (72, 0.559750497341156)]


# STEP 7: Re-Ranker (Diversity + Deduplication)

In [33]:
def rerank_basic(results, top_k=10):
    seen = set()
    final = []
    
    for user, score in results:
        if user not in seen:
            final.append((user, score))
            seen.add(user)
        
        if len(final) >= top_k:
            break
    
    return final

In [34]:
final_results = rerank_basic(l2_results)

print("Final results:", len(final_results))
print("Recommendations:", final_results)

Final results: 10
Recommendations: [(28, 0.5719471573829651), (20, 0.5696403980255127), (111, 0.5648233890533447), (35, 0.5603176355361938), (72, 0.559750497341156), (62, 0.5593438148498535), (277, 0.5585928559303284), (92, 0.5579479336738586), (156, 0.5575478672981262), (42, 0.5568864345550537)]


In [35]:
def rerank_diversity(G, user, results, top_k=10):
    final = []
    selected_neighbors = []
    
    for candidate, score in results:
        cand_neighbors = set(G.neighbors(candidate))
        
        # Penalize similarity with already selected users
        penalty = 0
        for sel in selected_neighbors:
            overlap = len(cand_neighbors & sel)
            penalty += overlap
        
        adjusted_score = score - 0.001 * penalty
        
        final.append((candidate, adjusted_score, cand_neighbors))
    
    # sort again after penalty
    final.sort(key=lambda x: x[1], reverse=True)
    
    output = []
    for cand, score, neigh in final:
        output.append((cand, score))
        selected_neighbors.append(neigh)
        
        if len(output) >= top_k:
            break
    
    return output

In [36]:
final_results = rerank_diversity(G, user, l2_results)

print("Final diversified results:", final_results)

Final diversified results: [(28, 0.5719471573829651), (20, 0.5696403980255127), (111, 0.5648233890533447), (35, 0.5603176355361938), (72, 0.559750497341156), (62, 0.5593438148498535), (277, 0.5585928559303284), (92, 0.5579479336738586), (156, 0.5575478672981262), (42, 0.5568864345550537)]
